creating pivot view sorted by region for the sampled_par.xlsx files, then plot by 
regions: st johns, avalon, western, central

In [ ]:
import os
import openpyxl
import pandas as pd
import matplotlib.pyplot as plt
import math
import datetime
import matplotlib.patches as mpatches
import re

In [ ]:
increment = 9
func = 'mean'

In [ ]:
folder = r"C:\Users\CAMG038492\OneDrive - WSP O365\Documents\Climate Data\NF Power GIS\Output Samples\uploaded"
paralist = [t[8:-5] for t in os.listdir(folder) if t.startswith("sampled_") and not t.endswith(".csv")]
# todo later: flood prsn-50mm
para = paralist[increment:][0]
print(para)
path = os.path.join(folder, "sampled_"+para+".xlsx")
path

In [ ]:
sheets = pd.read_excel(path, sheet_name=None)

In [ ]:
cdf = pd.DataFrame()
for name, df in sheets.items():
    if name != "metadata":
        for col in df.columns:
            if col not in cdf.columns:
                cdf[col] = df[col]
cdf.head()

In [ ]:
# region filtering
regions = {
    "St Johns" : ["St. John's", "St.John's"],
    "Avalon" : ['Avalon', 'Eastern', 'Burin'],
    "Central" : ['Central', 'Bonavista', 'Gander', 'Grand Falls'],
    "Western" : ['Western', 'Corner Brook', 'Stephenville']
}
mapping = {}
for v, synonyms in regions.items():
    for synonym in synonyms:
        mapping[synonym] = v  # map each synonym to the standard name

In [ ]:
cdf['Region_summarized'] = cdf['Region'].map(mapping)
cdf

In [ ]:
canrcm4list = ['humidex', 'pr-50mm','prsn-50mm', 'pr','prsn']
awkcanrcm4list = ['drydays','humidex30','pr50','prsn50','prfr']

In [ ]:
datacols = [t for t in cdf.columns if t.startswith(para) and not "45_" in t and not "126_" in t]
if para == 'sfcWindmax':
    datacols = [s for s in datacols if "anwindspeed63" in s]
if para in canrcm4list:
    datacols = [s for s in datacols if 'annsum' not in s and 'annstd' not in s]

In [ ]:
if func == 'mean': grouped = cdf.groupby('Region_summarized')[datacols].mean()
if func == 'min': grouped = cdf.groupby('Region_summarized')[datacols].min()
if func == 'max': grouped = cdf.groupby('Region_summarized')[datacols].max()
grouped.transpose()
grouped.transpose().to_csv(os.path.join(folder,"plots",f"grouped_region{func}_{para}.csv"))

In [ ]:
if whiskers:=False:
    df_mean = grouped = cdf.groupby('Region_summarized')[datacols].mean()
    df_min  = grouped = cdf.groupby('Region_summarized')[datacols].min()
    df_max  = grouped = cdf.groupby('Region_summarized')[datacols].max()

    regions = [c for c in df_mean.columns if c != 'Unnamed: 0']

    mean_vals = df_mean[regions]
    min_vals  = df_min[regions]
    max_vals  = df_max[regions]

    # Compute whisker lengths
    lower_error = mean_vals - min_vals
    upper_error = max_vals - mean_vals
    yerr = [lower_error.values, upper_error.values]

    # Plot
    fig, ax = plt.subplots(figsize=(8,5))
    ax.bar(regions, mean_vals.values, yerr=yerr, capsize=5, color='skyblue', edgecolor='black')
    ax.set_ylabel('Value')
    ax.set_title('Region Means with Min/Max Whiskers')
    plt.show()

In [ ]:
low = [v for v in datacols if 'p5_' in v or 'p10' in v]
mid = [v for v in datacols if 'p50' in v]
hig = [v for v in datacols if 'p95' in v or 'p90' in v]
mid2 = [v for v in datacols if v not in low and v not in hig]
grouplow = cdf.groupby('Region_summarized')[low].mean().transpose()
groupmid = cdf.groupby('Region_summarized')[mid].mean().transpose()
grouphig = cdf.groupby('Region_summarized')[hig].mean().transpose()
groupmid2 = cdf.groupby('Region_summarized')[mid2].mean().transpose()

In [ ]:
groupmid2.plot()
groupmid2.columns

In [ ]:
attempt1 = False
if attempt1:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    data = [grouplow,groupmid,grouphig]
    heads = ['p5 p10','p50','p90 p95']
    for t in range(3):
        axes[t].plot(data[t])
        axes[t].set_title(heads[t])
        axes[t].set_xticks(data[t].index)
        axes[t].set_xticklabels(data[t].index, rotation=90)
        #axes[t].legend(title='Region', loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# Dark mode style
plt.style.use('dark_background')
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': 'white',
    'figure.facecolor': '#222222',
    'axes.facecolor': '#222222',
    'grid.color': '#555555'
})

In [ ]:
df = grouped
regions = grouped.index.tolist()
numeric_df = grouped

order_map = {'min': 0, 'p5': 0, 'p10': 1, 'p50': 2, 'avg':3, 'p90': 4, 'p95': 5, 'max':6,'': 7}
labels_data = []
seen = []
for col in numeric_df.columns:
    if para == 'flashrate':
        scen = 'CESM2' if 'CESM2' in col else ('UKESM' if 'UKESM' in col else ('GISS' if 'GISS' in col else ''))
    elif para in canrcm4list:
        scen = 'min' if 'annmin' in col else ('max' if 'annmax' in col else ('p10' if 'p10' in col else ('p90' if 'p90' in col else('p50' if 'annmean' in col else ''))))
    elif 'p95' in col: scen = 'p95'
    elif 'p90' in col: scen = 'p90'
    elif 'p50' in col: scen = 'p50'
    elif 'p10' in col: scen = 'p10'
    elif 'p5_' in col: scen = 'p5'
    else:
        if para in awkcanrcm4list: scen = 'mean'
        else: scen = ''
    try: date = re.search(r'\d{4}-\d{4}', col).group(0)
    except: date = re.search(r'\d{6}-\d{6}', col).group(0)
    label = f'{scen}_{date}' if scen else date
    if label in seen:
        raise ValueError(f"duplicate found {label} for {col}, rerun")
    else:
        seen.append(label)
    labels_data.append({'col': col, 'scenario': scen, 'date': date, 'label': label, 'order': order_map.get(scen, 3)})

labels_data_sorted = sorted(labels_data, key=lambda x: (x['date'], x['order']))
sorted_cols = [d['col'] for d in labels_data_sorted]
sorted_labels = [d['label'] for d in labels_data_sorted]


In [ ]:
c_default = '#CCCCCC'
c_p5 = '#E69F00'
c_p50 = '#56B4E9'
c_p95 = '#009E73'
sorted_colors = [c_p95 if d['scenario'] == 'p95' or d['scenario'] == 'p90' or d['scenario'] == 'max'
                 else c_p50 if d['scenario'] == 'p50' or d['scenario'] == 'mean'
                 else c_p5 if d['scenario'] == 'p5' or d['scenario'] == 'p10' or d['scenario'] == 'min'
                 else c_default for d in labels_data_sorted]

In [ ]:
n = len(df)
rows = int(math.ceil(math.sqrt(n)))
cols = int(math.ceil(n / rows))
fig, axes = plt.subplots(rows, cols, figsize=(cols*8, rows*6))
axes = axes.flatten() if n > 1 else [axes]

for i in range(n):
    vals = numeric_df.iloc[i][sorted_cols].values
    bars = axes[i].barh(sorted_labels, vals, color=sorted_colors)
    axes[i].invert_yaxis()  # first label at top
    axes[i].set_title(regions[i])
    axes[i].set_xlabel('Value')
    #axes[i].set_ylabel('Scenario & Date Range')
    axes[i].grid(axis='x', linestyle='--', alpha=0.5)
    for bar in bars:
        width = bar.get_width()
        y = bar.get_y() + bar.get_height() / 2.
        axes[i].text(width, y, f'{width:.3f}', va='center', ha='left', fontsize=9, fontweight='bold', color='white')

for j in range(n, len(axes)):
    fig.delaxes(axes[j])

legend_handles = [
    mpatches.Patch(color=c_p5, label='p5'),
    mpatches.Patch(color=c_p50, label='p50'),
    mpatches.Patch(color=c_p95, label='p95'),
    mpatches.Patch(color=c_default, label='other')
]
leg = fig.legend(handles=legend_handles, loc='upper right', title='Scenario', fontsize=12, title_fontsize=14)
for text in leg.get_texts():
    text.set_color('white')
leg.get_title().set_color('white')

plt.tight_layout(rect=[0, 0, 0.95, 0.95])
plt.suptitle(para + "  " + str(datetime.datetime.now()))
#plt.savefig(f"plotted_region{func}_{para}.png")
plt.savefig(os.path.join(folder,"plots",f"plotted_region{func}_{para}.png"))
plt.show()


In [ ]:
increment += 1
print(f"next {paralist[increment:][0]}")